# Nexus — API Validation & Technical Synthesis Notebook

**Purpose.** This notebook is the authoritative reference for the Nexus derivatives terminal. It does two jobs at once:

1. **Live-validates every backend endpoint** — health, market data, zones, order flow, alpha engine, heatmap, AI brief, risk, macro, Binance futures, technicals — so a single run tells you what is green / degraded / down.
2. **Explains the synthesis** — for every indicator, signal, and data stream, *what it is, what it measures, and how a derivatives trader actually uses it*.

**Prereqs.**
- Backend running at `http://127.0.0.1:8001` (`python backend/main.py` or `Nexus.bat`).
- Ollama running locally with `gemma4:e4b` pulled for the AI brief section.

Run top-to-bottom. Each cell is idempotent and hits live data.

In [2]:
import time, json, statistics
from typing import Any, Dict, List, Optional
import requests

API = "http://127.0.0.1:8001"
SYMBOL = "BTCUSDT"

RESULTS: List[Dict[str, Any]] = []

def probe(method: str, path: str, *, params=None, json_body=None, timeout=15, note: str = "") -> Dict[str, Any]:
    url = f"{API}{path}"
    t0 = time.perf_counter()
    try:
        r = requests.request(method, url, params=params, json=json_body, timeout=timeout)
        ms = (time.perf_counter() - t0) * 1000
        ok = r.status_code < 500
        sample = None
        try:
            data = r.json()
            if isinstance(data, dict):
                sample = {k: ("<"+type(v).__name__+">" if isinstance(v, (list, dict)) else v) for k, v in list(data.items())[:6]}
            elif isinstance(data, list):
                sample = f"list[{len(data)}]"
        except Exception:
            sample = r.text[:120]
        row = {"method": method, "path": path, "status": r.status_code, "ms": round(ms, 1), "ok": ok, "note": note, "sample": sample}
    except Exception as e:
        row = {"method": method, "path": path, "status": "ERR", "ms": None, "ok": False, "note": note, "sample": str(e)[:120]}
    RESULTS.append(row)
    flag = "✓" if row["ok"] else "✗"
    print(f"{flag} [{row['status']:>3}] {method:<5} {path:<45} {row['ms']!s:>6} ms  {note}")
    return row

print(f"Target: {API}  |  Symbol: {SYMBOL}")

Target: http://127.0.0.1:8001  |  Symbol: BTCUSDT


## 1 · Health & Session
**`GET /api/health`** — heartbeat. Returns `{status, version, timestamp}`. First thing Electron's splash screen polls; if this 500s, nothing else boots.

In [3]:
probe("GET", "/api/health", note="liveness")

✓ [200] GET   /api/health                                     10.0 ms  liveness


{'method': 'GET',
 'path': '/api/health',
 'status': 200,
 'ms': 10.0,
 'ok': True,
 'note': 'liveness',
 'sample': {'status': 'ok',
  'version': '0.3.0',
  'websockets': '<dict>',
  'symbols': '<list>',
  'gemma4_model': 'gemma4:e4b',
  'blofin_connected': True}}

## 2 · Binance Futures Market Data
These are the endpoints the Trading tab's hero hits every time you type in the pair search or the active symbol changes.

| Endpoint | What it returns | Trader use-case |
|---|---|---|
| `GET /api/symbols/search?q=<txt>` | USDT-M PERPETUAL symbols matching query, majors ranked first | Autocomplete for the leverage-pair input. |
| `GET /api/ticker/{symbol}` | `last_price`, `price_change_pct`, `high_24h`, `low_24h`, `volume_24h`, `quote_volume_24h`, `trades_24h` | Dynamic ticker strip in the header — replaces the old static BTC badge. |
| `GET /api/klines/{symbol}?interval=1h&limit=500` | OHLCV candles, 2s TTL cache | Primary feed for the candlestick chart + all client-side overlays. |

In [4]:
probe("GET", "/api/symbols/search", params={"q": "btc"}, note="autocomplete")
probe("GET", f"/api/ticker/{SYMBOL}", note="24h stats")
probe("GET", f"/api/klines/{SYMBOL}", params={"interval": "1h", "limit": 200}, note="OHLCV 1h")
probe("GET", f"/api/klines/{SYMBOL}", params={"interval": "5m", "limit": 100}, note="OHLCV 5m")

✓ [200] GET   /api/symbols/search                            340.1 ms  autocomplete
✓ [200] GET   /api/ticker/BTCUSDT                             86.2 ms  24h stats
✓ [200] GET   /api/klines/BTCUSDT                            123.9 ms  OHLCV 1h
✓ [200] GET   /api/klines/BTCUSDT                            122.5 ms  OHLCV 5m


{'method': 'GET',
 'path': '/api/klines/BTCUSDT',
 'status': 200,
 'ms': 122.5,
 'ok': True,
 'note': 'OHLCV 5m',
 'sample': {'symbol': 'BTCUSDT',
  'interval': '5m',
  'count': 100,
  'candles': '<list>'}}

## 3 · Technical Indicators — `/api/indicators/{symbol}`

Returns `{latest, pivots, series}`. All indicators computed server-side from Binance USDT-M klines so the wire payload stays small and the Trading tab is GPU-light.

### Synthesis — what each indicator means

- **RSI 14** (Wilder's smoothing). Momentum oscillator 0–100. `>70` overbought, `<30` oversold, `50` is the bull/bear pivot. In trending markets RSI stays pinned `>50` (uptrend) or `<50` (downtrend); divergence vs. price is a counter-trend warning.
- **EMA 50 / EMA 200**. Classic trend filter. `EMA50 > EMA200` = bullish regime ("golden cross" on the transition); the reverse is a death cross. Used in `ema_trend` and `ema_cross` fields.
- **Bollinger Bands (20, 2σ)**. Volatility envelope around SMA 20. Price hugging the upper band in an uptrend = strength, not reversal. **Band width** (`bb_width_pct`) compresses before breakouts — the "squeeze".
- **MACD (12, 26, 9)**. Trend-following momentum: `macd` line crossing `macd_signal` triggers bias flips (`macd_bias`). The histogram `macd_hist` is early warning — it peaks before price.
- **Stochastic (14, 3, 3)**. Faster oscillator; `%K`/`%D` crosses in extreme zones (`>80` / `<20`) are the tactical entries when the higher-timeframe trend agrees.
- **ATR 14**. Average True Range. Dollar volatility — use it to size stops (`1.5 × ATR` is a common stop distance) and to normalize position sizes across symbols.
- **ADX 14**. Trend strength, direction-agnostic. `<20` = chop (oscillator strategies), `>25` = trending (trend strategies, breakouts). `adx_regime` collapses this into a label the Alpha tab consumes.
- **Classic pivots** (P, R1-R3, S1-S3) from prior bar's H/L/C. Intraday S/R scaffolding; magnet levels for algos.

### Confluence score
The Trading tab's right rail combines RSI + EMA trend + MACD bias + Stoch + ADX into a single **-100…+100** score. Above +60 = strong bull alignment; below -60 = strong bear; mid-band is noise.

In [5]:
r = probe("GET", f"/api/indicators/{SYMBOL}", params={"interval": "1h", "limit": 500}, note="full TA pack")
try:
    data = requests.get(f"{API}/api/indicators/{SYMBOL}", params={"interval": "1h", "limit": 500}, timeout=15).json()
    latest = data.get("latest", {})
    print("\nLATEST READOUT")
    for k in ["rsi_14", "ema_50", "ema_200", "ema_trend", "ema_cross", "bb_upper", "bb_middle", "bb_lower", "bb_width_pct", "macd", "macd_signal", "macd_hist", "macd_bias", "stoch_k", "stoch_d", "atr_14", "adx_14", "adx_regime"]:
        v = latest.get(k)
        if isinstance(v, float):
            print(f"  {k:<16} {v:>12.4f}")
        else:
            print(f"  {k:<16} {v}")
    piv = data.get("pivots", {})
    if piv:
        print("\nPIVOTS")
        for k, v in piv.items():
            print(f"  {k:<4} {v}")
except Exception as e:
    print("indicators readout failed:", e)

✓ [200] GET   /api/indicators/BTCUSDT                        147.2 ms  full TA pack

LATEST READOUT
  rsi_14                62.1100
  ema_50             74559.6093
  ema_200            72944.6173
  ema_trend        bullish
  ema_cross           1614.9920
  bb_upper           75602.1710
  bb_middle          74740.0150
  bb_lower           73877.8590
  bb_width_pct           2.3071
  macd                 134.6415
  macd_signal           92.5597
  macd_hist             42.0818
  macd_bias        bullish
  stoch_k               86.6400
  stoch_d               71.5300
  atr_14               414.2526
  adx_14                18.9600
  adx_regime       ranging

PIVOTS
  r2   76089.466667
  r1   75798.533333
  pivot 75375.066667
  s1   75084.133333
  s2   74660.666667


## 4 · Market Structure — OI, Funding, L/S, CVD

| Endpoint | Synthesis |
|---|---|
| `GET /api/oi/{symbol}` | **Open Interest** — notional of live futures contracts. Rising OI + rising price = new longs (trend continuation). Rising OI + falling price = new shorts. Falling OI = positions closing (exhaustion). |
| `GET /api/funding/{symbol}` | **Funding rate** — perp-vs-spot tether. Persistent positive funding = longs pay shorts (crowded long); negative = crowded short. Extremes front-run squeezes. |
| `GET /api/lsratio/{symbol}` | **Long/Short ratio** (global + top-trader) — positioning tilt. Top-trader L/S is the smart-money proxy; retail L/S is the contrarian signal. |
| `GET /api/cvd/{symbol}` | **Cumulative Volume Delta** — signed aggressor flow (market-buys minus market-sells). CVD divergence from price is the institutional accumulation/distribution tell. |

In [6]:
probe("GET", f"/api/oi/{SYMBOL}", note="open interest")
probe("GET", f"/api/funding/{SYMBOL}", note="funding")
probe("GET", f"/api/lsratio/{SYMBOL}", note="long/short")
probe("GET", f"/api/cvd/{SYMBOL}", note="cumulative delta")

✓ [200] GET   /api/oi/BTCUSDT                               3777.5 ms  open interest
✓ [200] GET   /api/funding/BTCUSDT                          4319.8 ms  funding
✓ [200] GET   /api/lsratio/BTCUSDT                          2482.5 ms  long/short
✓ [200] GET   /api/cvd/BTCUSDT                                 4.4 ms  cumulative delta


{'method': 'GET',
 'path': '/api/cvd/BTCUSDT',
 'status': 200,
 'ms': 4.4,
 'ok': True,
 'note': 'cumulative delta',
 'sample': {'symbol': 'BTCUSDT', 'timeframes': '<dict>'}}

## 5 · Alpha Engine — `/api/alpha/{symbol}`
Weighted composite of 8 signals. Each has a direction (`bullish`/`bearish`/`neutral`), magnitude, and confidence. The `composite_score` is the agreement-weighted blend; `agreement_ratio` tells you how many of the 8 agree.

- **OFI** — Order Flow Imbalance; aggressor pressure at the touch.
- **VWAP Deviation** — price distance from session VWAP normalized by ATR.
- **Funding Arbitrage** — exploits funding extremes vs. spot basis.
- **Cross-Exchange Spread** — Binance vs. BloFin/Deribit basis dislocation.
- **Liquidation Cascade** — leveraged-position heatmap proximity.
- **Delta Divergence** — CVD vs. price divergence.
- **Smart Money Flow** — top-trader L/S ratio deltas + whale prints.
- **Volatility Regime** — ATR / BB-width regime classifier switching between trend and mean-revert models.

In [7]:
probe("GET", f"/api/alpha/{SYMBOL}", note="composite signal")

✓ [200] GET   /api/alpha/BTCUSDT                            3186.1 ms  composite signal


{'method': 'GET',
 'path': '/api/alpha/BTCUSDT',
 'status': 200,
 'ms': 3186.1,
 'ok': True,
 'note': 'composite signal',
 'sample': {'symbol': 'BTCUSDT',
  'composite_score': 0.0,
  'composite_direction': 'neutral',
  'composite_confidence': 0.115,
  'signals': '<list>',
  'signal_count': 8}}

## 6 · Liquidity Map & Order Flow
- **`GET /api/heatmap/{symbol}`** — discretized liquidation clusters (price × size). The magnet zones; price tends to seek these before reversing.
- **`GET /api/orderflow/{symbol}`** — aggregated buy/sell volume, trade flow ratio, OI change over rolling windows. Feeds the Order Flow tab.

In [8]:
probe("GET", f"/api/heatmap/{SYMBOL}", note="liquidation clusters")
probe("GET", f"/api/orderflow/{SYMBOL}", note="aggressor flow")

✓ [200] GET   /api/heatmap/BTCUSDT                            52.5 ms  liquidation clusters
✓ [200] GET   /api/orderflow/BTCUSDT                          32.2 ms  aggressor flow


{'method': 'GET',
 'path': '/api/orderflow/BTCUSDT',
 'status': 200,
 'ms': 32.2,
 'ok': True,
 'note': 'aggressor flow',
 'sample': {'cvd': '<dict>',
  'absorption': '<dict>',
  'large_trades': '<list>',
  'trade_flow_ratio': 0.5,
  'volume_profile': '<dict>',
  'oi_change_1h': 0.196}}

## 7 · Zones & Alerts
- **`GET /api/zones/{symbol}`** — detected supply/demand zones with tier (S1/S2/S3), status (watching/active/invalidated), and price center.
- **`GET /api/zones/watchlist`** — cross-symbol watchlist.
- **`GET /api/alerts?limit=50`** — recent alert history (zone breaches, funding extremes, whale prints, AI triggers).
- **`POST /api/alerts/telegram/test`** — sanity-check bot delivery.
- **`POST /api/zones/alert`** — manually arm a zone alert.

In [9]:
probe("GET", f"/api/zones/{SYMBOL}", note="zones")
probe("GET", "/api/zones/watchlist", note="watchlist")
probe("GET", "/api/alerts", params={"limit": 20}, note="alert history")

✓ [200] GET   /api/zones/BTCUSDT                               7.3 ms  zones
✓ [200] GET   /api/zones/watchlist                            16.7 ms  watchlist
✓ [200] GET   /api/alerts                                      6.3 ms  alert history


{'method': 'GET',
 'path': '/api/alerts',
 'status': 200,
 'ms': 6.3,
 'ok': True,
 'note': 'alert history',
 'sample': {'alerts': '<list>'}}

## 8 · News & Macro
- **`GET /api/news`** — aggregated crypto news with sentiment labels. Feeds the Alerts tab News Feed and the AI synthesis.
- **`GET /api/macro/calendar`** — upcoming macro events (FOMC, CPI, NFP) with expected impact.
- **`GET /api/macro/status`** — current macro regime flags (DXY bias, yield curve, risk-on/off).
- **`GET /api/deribit/options/{currency}`** — BTC/ETH options snapshots (IV, OI by strike, put/call, max pain).
- **`GET /api/deribit/trades/{instrument}`** — recent option prints.

In [10]:
probe("GET", "/api/news", note="news feed")
probe("GET", "/api/macro/calendar", note="macro events")
probe("GET", "/api/macro/status", note="macro regime")
probe("GET", "/api/deribit/options/BTC", note="BTC options")
probe("GET", "/api/deribit/options/ETH", note="ETH options")

✓ [200] GET   /api/news                                     9287.6 ms  news feed
✓ [200] GET   /api/macro/calendar                              4.4 ms  macro events
✓ [200] GET   /api/macro/status                               16.1 ms  macro regime
✓ [200] GET   /api/deribit/options/BTC                      3388.2 ms  BTC options
✓ [200] GET   /api/deribit/options/ETH                      3487.1 ms  ETH options


{'method': 'GET',
 'path': '/api/deribit/options/ETH',
 'status': 200,
 'ms': 3487.1,
 'ok': True,
 'note': 'ETH options',
 'sample': {'put_call_ratio': '<dict>',
  'max_pain': '<dict>',
  'perpetual_funding': '<dict>'}}

## 9 · AI Brief — Gemma 4 synthesis

The brief generator makes **two** local LLM calls:
1. **Main brief** — structured market assessment (regime, bias, key risks, zones to watch).
2. **News synthesis** — 2–4 sentence narrative paragraph distilling the top 15 headlines into the *dominant storyline + contradictions + what a derivatives trader should care about*.

`POST /api/ai/brief` generates both; `GET /api/ai/last-brief` returns the cached last run (so the UI renders instantly on tab-open).

In [11]:
probe("GET", "/api/ai/last-brief", note="cached brief")
# Full generation — takes 10-60s depending on model warmth.
print("\nGenerating fresh brief (may take ~30s)…")
r = probe("POST", "/api/ai/brief", timeout=120, note="fresh synthesis")
try:
    data = requests.post(f"{API}/api/ai/brief", timeout=120).json()
    print("\n--- NEWS SYNTHESIS ---\n")
    print(data.get("news_synthesis") or "(empty)")
    print("\n--- BRIEF (first 800 chars) ---\n")
    print((data.get("brief") or "")[:800])
    print(f"\nsentiment={data.get('news_sentiment')}  headlines={data.get('news_count')}")
except Exception as e:
    print("brief readout failed:", e)

✓ [200] GET   /api/ai/last-brief                               7.2 ms  cached brief

Generating fresh brief (may take ~30s)…
✓ [200] POST  /api/ai/brief                                 26097.2 ms  fresh synthesis

--- NEWS SYNTHESIS ---

[Gemma4 error: HTTP 500]

--- BRIEF (first 800 chars) ---

[Gemma4 error: HTTP 500]

sentiment=neutral  headlines=55


## 10 · Risk
- **`POST /api/risk/kelly`** — fractional-Kelly position sizer given edge, win-rate, reward/risk.
- **`GET /api/risk/kelly/{symbol}`** — symbol-specific Kelly using live alpha stats.
- **`GET /api/risk/leverage`** — current leverage across open positions.
- **`GET /api/risk/margin`** — margin usage / maintenance buffer.
- **`POST /api/risk/simulate`** — Monte-Carlo equity-curve sim for a proposed trade.

In [12]:
probe("POST", "/api/risk/kelly", json_body={"win_rate": 0.55, "reward_risk": 1.8, "edge": 0.05}, note="kelly sizer")
probe("GET", f"/api/risk/kelly/{SYMBOL}", note="symbol kelly")
probe("GET", "/api/risk/leverage", note="leverage")
probe("GET", "/api/risk/margin", note="margin")
probe("POST", "/api/risk/simulate", json_body={"symbol": SYMBOL, "side": "long", "size_usd": 1000, "stop_pct": 1.5, "target_pct": 3.0, "n": 500}, note="monte carlo")

✓ [422] POST  /api/risk/kelly                                 13.9 ms  kelly sizer
✓ [200] GET   /api/risk/kelly/BTCUSDT                         14.2 ms  symbol kelly
✓ [200] GET   /api/risk/leverage                            1601.8 ms  leverage
✓ [200] GET   /api/risk/margin                              2705.8 ms  margin
✓ [422] POST  /api/risk/simulate                               9.2 ms  monte carlo


{'method': 'POST',
 'path': '/api/risk/simulate',
 'status': 422,
 'ms': 9.2,
 'ok': True,
 'note': 'monte carlo',
 'sample': {'detail': '<list>'}}

## 11 · BloFin (execution venue)
Read-only probes — do **not** fire live orders from the notebook.
- **`GET /api/blofin/balance`** — account equity & margin.
- **`GET /api/blofin/positions`** — live positions.
- **`GET /api/blofin/orders`** — open orders.

In [13]:
probe("GET", "/api/blofin/balance", note="equity")
probe("GET", "/api/blofin/positions", note="positions")
probe("GET", "/api/blofin/orders", note="open orders")

✓ [200] GET   /api/blofin/balance                            958.1 ms  equity
✓ [200] GET   /api/blofin/positions                          101.2 ms  positions
✓ [200] GET   /api/blofin/orders                              97.9 ms  open orders


{'method': 'GET',
 'path': '/api/blofin/orders',
 'status': 200,
 'ms': 97.9,
 'ok': True,
 'note': 'open orders',
 'sample': {'orders': '<list>'}}

## 12 · Summary Table

In [14]:
ok = sum(1 for r in RESULTS if r["ok"])
bad = len(RESULTS) - ok
latencies = [r["ms"] for r in RESULTS if isinstance(r["ms"], (int, float))]
p50 = statistics.median(latencies) if latencies else 0
p95 = sorted(latencies)[int(len(latencies) * 0.95) - 1] if len(latencies) >= 5 else max(latencies, default=0)

print(f"ENDPOINTS  {ok}/{len(RESULTS)} OK  |  p50={p50:.0f}ms  p95={p95:.0f}ms")
print()
print(f"{'STATUS':<7} {'MS':>6}  {'METHOD':<5} {'PATH':<45}  NOTE")
print("-" * 90)
for r in RESULTS:
    flag = "OK" if r["ok"] else "FAIL"
    ms = f"{r['ms']:.0f}" if isinstance(r["ms"], (int, float)) else "-"
    print(f"{flag:<7} {ms:>6}  {r['method']:<5} {r['path']:<45}  {r['note']}")

failures = [r for r in RESULTS if not r["ok"]]
if failures:
    print("\nFAILURES:")
    for f in failures:
        print(f"  {f['method']} {f['path']} -> {f['status']}  {f['sample']}")

ENDPOINTS  31/31 OK  |  p50=101ms  p95=4320ms

STATUS      MS  METHOD PATH                                           NOTE
------------------------------------------------------------------------------------------
OK          10  GET   /api/health                                    liveness
OK         340  GET   /api/symbols/search                            autocomplete
OK          86  GET   /api/ticker/BTCUSDT                            24h stats
OK         124  GET   /api/klines/BTCUSDT                            OHLCV 1h
OK         122  GET   /api/klines/BTCUSDT                            OHLCV 5m
OK         147  GET   /api/indicators/BTCUSDT                        full TA pack
OK        3778  GET   /api/oi/BTCUSDT                                open interest
OK        4320  GET   /api/funding/BTCUSDT                           funding
OK        2482  GET   /api/lsratio/BTCUSDT                           long/short
OK           4  GET   /api/cvd/BTCUSDT                               c

---
## Appendix · Field glossary

Human labels map to snake_case backend fields via `frontend/src/lib/labels.ts`. Any new field added to the backend must also get an entry there, otherwise the UI will title-case the raw key.

**Core market data**: `mark_price`, `index_price`, `funding_rate`, `perpetual_funding`, `weighted_rate_pct`, `oi_change_pct`, `oi_change_1h`, `ls_ratio`, `top_trader_ls_ratio`, `put_call_ratio`, `max_pain`, `quote_volume_24h`, `volume_24h`, `trades_24h`, `high_24h`, `low_24h`, `open_24h`, `change_24h_pct`.

**Alpha signals**: `ofi`, `vwap_deviation`, `funding_arb`, `cross_exchange_spread`, `liquidation_cascade`, `delta_divergence`, `smart_money_flow`, `vol_regime`, plus composite fields `composite_score`, `composite_direction`, `agreement_ratio`, `net_flow`, `recent_whales`.

**Technicals**: `rsi_14`, `ema_50`, `ema_200`, `ema_cross`, `ema_trend`, `bb_upper`, `bb_middle`, `bb_lower`, `bb_width_pct`, `macd`, `macd_signal`, `macd_hist`, `macd_bias`, `stoch_k`, `stoch_d`, `atr_14`, `adx_14`, `adx_regime`.

**Order flow**: `trade_flow_ratio`, `buy_volume`, `sell_volume`, `cvd`.

**Zones**: `price_center`, `zone_type`, `tier`, `status`.

**Regime**: `regime`, `confidence`.